# Phase 5: a never-trained motion, ready for the robot

The MVP path, with no change to the robot's software: the general tracker's weights plus **one motion
it never trained on**, exported with pupper-mjlab's own `export_trick` and written as a normal trick
folder (`policy.json` + `trick.yaml`). On the robot, the launch file loads one controller per trick at
startup, so after a restart Gemini can play it like any course trick. That's the plan's option (i), done
with a restart instead of runtime loading. Runtime loading is for the TA conversation.

1. **Pick the motion:** a held-out test motion that v2 tracked well in Phase 4, or a brand-new request.
2. **Export and check** (`kflib/deploy.py`): the same deploy contract as a course trick, the motion
   embedded is the one we picked, the reference stays well away from the tilt e-stop, and the robot's
   own loader (`robot/tricks.py`) accepts the folder.
3. **Simulate the exact file** that would go on the robot (`track.py`, json mode, plus plain replay).
4. **Package it** for the robot.

**Nothing here moves the robot.** The first run of a new trained policy needs **the TA's OK**, with
the robot on the ground, clear space, a hand near the e-stop, and you at the robot
(`final_project/phase5_ta_brief.md`).

In [ ]:
# @title Setup (about 2 minutes; a GPU runtime is faster, CPU works)
import os, sys, subprocess
COMMIT = "aa475e9a393a209b29fd5de7e0ee10f3de643230"
BRANCH = "final-project-kflib"  # @param {type:"string"}
TRACKER = "pgt_v2"  # @param {type:"string"}
if not os.path.exists("/content/pupper-mjlab"):
  !git clone -q https://github.com/cs123-stanford/pupper-mjlab /content/pupper-mjlab
!git -C /content/pupper-mjlab checkout -q {COMMIT}
!pip install -q -e /content/pupper-mjlab
sys.path.insert(0, "/content/pupper-mjlab/src")
if not os.path.exists("/content/gemini_pupper"):
  !git clone -q -b {BRANCH} https://github.com/MlsBli/gemini_pupper /content/gemini_pupper
!git -C /content/gemini_pupper pull -q
sys.path.insert(0, "/content/gemini_pupper/final_project")
os.chdir("/content/gemini_pupper/final_project")

from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/pupper_general_tracker")
TEST, RUNS = ROOT / "datasets" / "test_v1", ROOT / "runs"
ROBOT_TRICKS = ROOT / "robot_tricks"  # trick folders for the robot's tricks/ directory
from kflib.runs import latest_checkpoint
from kflib.testset import TestSet
ts = TestSet(TEST)
IT, CKPT = latest_checkpoint(RUNS, TRACKER)
LABEL = f"{TRACKER}@{IT + 1}"
print("tracker:", LABEL, CKPT)

## 1. Pick the motion

The plan's rule for a first robot run: a held-out motion that **held up well in simulation**. These are the
gentlest-tracked held-out motions from Phase 4 under this tracker: HOLDS UP, and plain replay also stayed
up. Pick one by id, or type a new request below instead.

In [ ]:
# @title Candidates from Phase 4 (held out, tracked well)
import json
by_id = {r["id"]: r for r in ts.rows()}
res = {}
for f in (TEST / "results").glob("*.json"):
  mid, label = f.stem.split("__")
  res.setdefault(mid, {})[label] = json.loads(f.read_text())
cands = [(mid, r[LABEL]) for mid, r in res.items() if LABEL in r and r[LABEL].get("verdict") == "HOLDS UP"
         and r.get("plain", {}).get("verdict") in ("HOLDS UP", "WOBBLES") and by_id.get(mid, {}).get("set") in ("4a", "4b")]
cands.sort(key=lambda c: c[1]["pos_cm"] / 3 + c[1]["tilt_deg"] / 10)
for mid, r in cands[:10]:
  print(f"{mid:22} {r['pos_cm']} cm {r['tilt_deg']} deg | plain {res[mid]['plain']['verdict']:8} | {by_id[mid]['request'][:70]}")

In [ ]:
# @title The motion: a test motion id, or a new request
MOTION_ID = "bench_bow_r0"  # @param {type:"string"}
NEW_REQUEST = ""  # @param {type:"string"}
TRICK_NAME = "pgt_bow"  # @param {type:"string"}
DESCRIPTION = "From standing, a playful bow: chest down, rear up, then back to standing."  # @param {type:"string"}
# DESCRIPTION is what Gemini reads when deciding to play it: say what the robot does.
if NEW_REQUEST.strip():
  from kflib.llm import LLM
  from kflib.motion import motion_from_dict
  from kflib.pipeline import request_to_motion
  rec = request_to_motion(NEW_REQUEST, LLM("colab"))
  assert rec["ok"], rec["reasons"]
  MOTION_NPZ = Path("/content") / f"{TRICK_NAME}.npz"
  motion_from_dict(rec["motion"]).save_npz(MOTION_NPZ)
  SOURCE = dict(request=NEW_REQUEST, tracker=LABEL)
else:
  MOTION_NPZ = ts.motions / f"{MOTION_ID}.npz"
  SOURCE = dict(test_motion=MOTION_ID, request=by_id[MOTION_ID]["request"], tracker=LABEL)
print(SOURCE)

## 2. Export and check

In [ ]:
# @title Export (pupper-mjlab's export_trick) and check (kflib/deploy.py)
from kflib.deploy import make_trick
info = make_trick(CKPT, MOTION_NPZ, TRICK_NAME, DESCRIPTION, ROBOT_TRICKS, SOURCE)
print(json.dumps(info, indent=1))
print((ROBOT_TRICKS / TRICK_NAME / "trick.yaml").read_text())

## 3. Simulate the exact file that would go on the robot

`policy.json` from the folder above, played by `track.py` (json mode), next to plain replay and the checkpoint
it came from. **Ready for the TA only if the exported file HOLDS UP.** Its paw lifts are checked too.

In [ ]:
# @title Simulate the exported policy.json
from types import SimpleNamespace
from kflib.track import run_many
from kflib.zeroshot import foot_tracking
DEST = ROBOT_TRICKS / TRICK_NAME
base = dict(id="motion", set="5", source="deploy", npz=str(DEST / "motion.npz"))  # id "motion": the reference is DEST/motion.npz
jobs = [dict(base, mode="json", policy=str(DEST / "policy.json"), label="exported"),
        dict(base, mode="checkpoint", policy=str(CKPT), label=LABEL), dict(base, mode="plain", label="plain")]
rows = run_many(jobs, DEST / "sim", workers=3)
for label in ("exported", LABEL, "plain"):
  ft = foot_tracking(rows, SimpleNamespace(motions=DEST), DEST / "sim", label=label)
  print(f"{label:16} paw lift: " + (f"{ft[0]['tracked_cm']} of {ft[0]['reference_cm']} cm ({ft[0]['ratio']:.2f})" if ft else "no paw lift in this motion"))
exported = next(r for r in rows if r["label"] == "exported")
ready = exported["verdict"] == "HOLDS UP"
print("\nREADY for the TA" if ready else "\nNOT ready: the exported file doesn't HOLD UP; pick another motion", exported["verdict"], exported["pos_cm"], "cm", exported["tilt_deg"], "deg")
(DEST / "sim_summary.json").write_text(json.dumps(dict(ready=ready, results=[{k: r.get(k) for k in ("label", "verdict", "pos_cm", "tilt_deg", "joint_rms_rad")} for r in rows]), indent=1))

In [ ]:
# @title (Optional) Watch the exported trick played by the policy (share link; stop the cell when done)
!python -m mjlab.tasks.tracking.config.pupper.play_motion {ROBOT_TRICKS / TRICK_NAME / 'motion.npz'} --checkpoint {CKPT} --viewer viser --share True --loops 2

## 4. Package it for the robot

Downloads `<TRICK_NAME>.zip` (policy.json, trick.yaml, motion.npz, the simulation results). **On the robot,
only with the TA's OK and you at the robot:** unzip it into the gemini_pupper repo's `tricks/` folder
(`tricks/<TRICK_NAME>/`), then restart Gemini Pupper (`robot/stop.sh`, then `robot/start.sh`); it loads
one controller per trick at startup. Then ask Gemini for it by what the description says.

In [ ]:
# @title Make the zip (and download it)
import shutil
assert json.loads((ROBOT_TRICKS / TRICK_NAME / "sim_summary.json").read_text())["ready"], "not ready: see the simulation above"
zip_path = shutil.make_archive(f"/content/{TRICK_NAME}", "zip", ROBOT_TRICKS, TRICK_NAME)
print("wrote", zip_path, "| also on Drive:", ROBOT_TRICKS / TRICK_NAME)
from google.colab import files
files.download(zip_path)